In [ ]:
import pandas as pd
import os
from typing import cast
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from data_preprocessing import create_train_test_val_sets, get_processed_df
from scipy.sparse import csr_matrix
import joblib


In [ ]:
#Create test train splits
x_mendeley, y_mendeley = get_processed_df(r"..\data\raw\Mendeley Dataset.csv")
x_kaggle, y_kaggle= get_processed_df(r"..\data\raw\dataset_phishing.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
kaggle_sets = create_train_test_val_sets(x_kaggle,y_kaggle, label_col="Label", test_size=0.2, n_splits=5)

### Tuning Classifiers

In [ ]:
#XGBoost

def optimize_xgboost(X, y, dataset, splits) -> XGBClassifier:
    """
    Returns a trained and tuned XGBClassifier
    
    Parameters:
    X: input data
    y: target variable
    dataset: which dataset is being used to tune the XGBClassifier
    splits: A list of tuples containing the splits for CV

    Returns:
    Tuned XGBClassifier
    """
    if hasattr(X, "sparse"):
        X = csr_matrix(X.sparse.to_coo())

    scale_weights = [1.0]
    counts = y.value_counts(normalize=True)
    scale_weights.append(counts[1]/counts[0]) 
    
    params = {
        'max_depth': [4, 5, 6, 8, 10],
        'gamma': [0.1, 0.2],
        'subsample': [0.6, 0.7],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.4],
        'n_estimators': [300, 500, 750, 1000, 1250],
        'scale_pos_weight': scale_weights
    }

    xgb = XGBClassifier(random_state=42)
    random_search = RandomizedSearchCV(xgb, param_distributions=params, random_state=42, cv=splits)
    random_search.fit(X, y)

    print('\n Best hyperparameters:')
    print(random_search.best_params_)

    return cast(XGBClassifier, random_search.best_estimator_)

os.makedirs("./models/phase_1", exist_ok=True)
print("Running hyperparameter tuning using Mendeley Dataset:")
xgboost_mendeley = optimize_xgboost(mendeley_sets["x_train"], mendeley_sets["y_train"], 'mendeley', mendeley_sets["cv_splits"])
joblib.dump(xgboost_mendeley, "./models/phase_1/xgboost_mendeley_no_fs.joblib")
# xgboost_mendeley = XGBClassifier(subsample=0.6, scale_pos_weight=0.93, n_estimators=750, max_depth=8, learning_rate=0.1, gamma=0.1)

print("Running hyperparameter tuning using kaggle Dataset:")
xgboost_kaggle = optimize_xgboost(kaggle_sets["x_train"], kaggle_sets["y_train"], 'kaggle', kaggle_sets["cv_splits"])
joblib.dump(xgboost_kaggle, "./models/phase_1/xgboost_kaggle_no_fs.joblib")
# xgboost_kaggle = XGBClassifier(subsample=0.6, scale_pos_weight=0.75, n_estimators=300,  max_depth=4, learning_rate=0.05, gamma=0.1)


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
def train_no_feature_selection(dataset, model):
    """
    Predict using the testset and print the confusion matrix and classification report

    Parameters:
    x_test: the test set for input variables
    y_test: the test set for the target variable
    model: the trained model
    """
    model.fit(dataset["x_train"], dataset["y_train"])
    y_test_pred = model.predict(dataset["x_test"])
    print("\nTest Set Performance")
    cm = confusion_matrix(dataset["y_test"], y_test_pred)
    ConfusionMatrixDisplay(cm).plot()
    print(classification_report(dataset["y_test"], y_test_pred))
    
xgboost_mendeley_baseline_no_fs = xgboost_mendeley
print('XGBoost Mendeley Results:')
train_no_feature_selection(mendeley_sets, xgboost_mendeley_baseline_no_fs)

xgboost_kaggle_baseline_no_fs = xgboost_kaggle
print('XGBoost kaggle Results:')
train_no_feature_selection(kaggle_sets, xgboost_kaggle_baseline_no_fs)

In [ ]:
import pandas as pd
import numpy as np
from copy import deepcopy

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression


def optimize_logistic_regression(X, y, splits):
    """
    Tune Logistic Regression using cross-validation splits.

    Parameters:
    X: input features
    y: target labels
    splits: list of (train_idx, val_idx) tuples for CV

    Returns:
    best fitted pipeline
    """
    params = {
        "model__C": [0.01, 0.1, 1, 10]
    }

    pipeline = Pipeline([
        ("scaler", MaxAbsScaler()),
        ("model", LogisticRegression(
            solver="saga",
            class_weight="balanced",
            max_iter=5000,
            tol=1e-3,
            random_state=42
        ))
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=splits,
        scoring="f1",
        n_jobs=1
    )

    grid_search.fit(X, y)

    print("\nBest hyperparameters:")
    print(grid_search.best_params_)
    print("Best CV F1:", grid_search.best_score_)

    return grid_search.best_estimator_

# --------------------------------------------------
# Logistic Regression with tuned value C=10
# --------------------------------------------------
# logreg_mendeley = Pipeline([
#     ("scaler", MaxAbsScaler()),
#     ("model", LogisticRegression(
#         C=10,
#         solver="saga",
#         class_weight="balanced",
#         max_iter=5000,
#         tol=1e-3,
#         random_state=42
#     ))
# ])


# logreg_kaggle = Pipeline([
#     ("scaler", MaxAbsScaler()),
#     ("model", LogisticRegression(
#         C=10,
#         solver="saga",
#         class_weight="balanced",
#         max_iter=5000,
#         tol=1e-3,
#         random_state=42
#     ))
# ])


# --------------------------------------------------
# Reuse existing evaluation function
# train_no_feature_selection(dataset, model)
# --------------------------------------------------

print("Running hyperparameter tuning using Mendeley Dataset:")
logreg_mendeley = optimize_logistic_regression(
    mendeley_sets["x_train"],
    mendeley_sets["y_train"],
    mendeley_sets["cv_splits"]
)

joblib.dump(logreg_mendeley, "./models/phase_1/logreg_mendeley_no_fs.joblib")

print("Running hyperparameter tuning using kaggle Dataset:")
logreg_kaggle = optimize_logistic_regression(
    kaggle_sets["x_train"],
    kaggle_sets["y_train"],
    kaggle_sets["cv_splits"]
)

joblib.dump(logreg_kaggle, "./models/phase_1/logreg_kaggle_no_fs.joblib")


print("Logistic Regression Mendeley Results:")
train_no_feature_selection(mendeley_sets, logreg_mendeley)

print("Logistic Regression kaggle Results:")
train_no_feature_selection(kaggle_sets, logreg_kaggle)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
import os
os.makedirs('./models/phase_1', exist_ok=True)

rf_mendeley = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    max_features='sqrt',
    class_weight=None,
    random_state=42,
    n_jobs=-1
)
joblib.dump(rf_mendeley, './models/phase_1/rf_mendeley_no_fs.joblib')

rf_kaggle = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=2,
    max_features='log2',
    class_weight=None,
    random_state=42,
    n_jobs=-1
)
joblib.dump(rf_kaggle, './models/phase_1/rf_kaggle_no_fs.joblib')

# --- Hyperparameter Tuning (commented out - takes a long time) ---
# from sklearn.model_selection import GridSearchCV
# param_grid = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [None, 10, 20],
#     'min_samples_split': [2, 5],
#     'max_features': ['sqrt', 'log2'],
#     'class_weight': [None, 'balanced']
# }
# rf_tuned_mendeley = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
#     param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1)
# rf_tuned_mendeley.fit(mendeley_sets['x_train'], mendeley_sets['y_train'])
# print('Best params (Mendeley):', rf_tuned_mendeley.best_params_)
# rf_tuned_kaggle = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
#     param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1)
# rf_tuned_kaggle.fit(kaggle_sets['x_train'], kaggle_sets['y_train'])
# print('Best params (Kaggle):', rf_tuned_kaggle.best_params_)

In [ ]:
print('Random Forest Mendeley Results:')
train_no_feature_selection(mendeley_sets, rf_mendeley)

print('Random Forest Kaggle Results:')
train_no_feature_selection(kaggle_sets, rf_kaggle)